# Self-RAG — let the model decide *whether* to use each retrieval

Naive RAG stuffs the top-k chunks into the prompt and hopes for the best. **Self-RAG** (Asai et al., 2023) adds two LLM-emitted control signals:

1. A per-chunk **relevance grade** before answering. Irrelevant chunks are dropped.
2. A post-hoc **support score** on the final answer. Unsupported answers are refused.

Result: better refusal behaviour on out-of-corpus questions, less hallucination, and a debug trail you can actually inspect.

In [1]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
print('LLM_CACHE_ONLY =', os.environ.get('LLM_CACHE_ONLY', '0'))


LLM_CACHE_ONLY = 1


In [2]:
from shared.embedders import cosine_topk, hash_embed
from shared.llm import Message, complete
from shared.loaders import load_corpus, load_golden_qa

MODEL = 'openai/gpt-4o-mini'
NS = '01-rag/06-self-rag'

DOCS = load_corpus()
QA = {q.id: q for q in load_golden_qa()}
doc_texts = [d.title + '. ' + d.abstract for d in DOCS]
doc_ids = [d.arxiv_id for d in DOCS]
doc_vecs = hash_embed(doc_texts, dims=256, seed=0)

def retrieve(q, k=3):
    qv = hash_embed([q], dims=256, seed=0)[0]
    idx, _ = cosine_topk(qv, doc_vecs, k=k)
    return [(doc_ids[i], doc_texts[i]) for i in idx]

## Stage 1 — per-chunk relevance grader

One LLM call per retrieved passage.

In [3]:
GRADER_SYS = (
    "You are a strict relevance grader. Given a question and a single passage, "
    "answer with EXACTLY one token: 'yes' if the passage is directly relevant to "
    "answering the question, otherwise 'no'."
)
def grade(q, passage):
    user = f'Question: {q}\n\nPassage: {passage}\n\nRelevant?'
    out = complete(model=MODEL, namespace=NS, messages=[
        Message(role='system', content=GRADER_SYS),
        Message(role='user', content=user),
    ]).content.strip().lower()
    return out.startswith('y')

## Stage 2 — supported answerer

Feed only the *relevant* passages, and demand a SUPPORT verdict.

In [4]:
ANSWERER_SYS = (
    "You are a careful research assistant. Use ONLY the relevant passages provided. "
    "After your answer, on a NEW line output 'SUPPORT: yes' if every claim is "
    "directly supported by the passages, otherwise 'SUPPORT: no'."
)
def answer(q, passages):
    ctx = '\n\n'.join(f'[{i}] {t}' for i, (_, t) in enumerate(passages)) or '(none)'
    user = f'Question: {q}\n\nRelevant passages:\n{ctx}\n\nAnswer:'
    out = complete(model=MODEL, namespace=NS, messages=[
        Message(role='system', content=ANSWERER_SYS),
        Message(role='user', content=user),
    ]).content
    body, _, tail = out.rpartition('SUPPORT:')
    return body.strip(), tail.strip().lower().startswith('y')

## End-to-end on four Q types

* `q01`, `q05`, `q11` are answerable from a single doc.
* `q27` is intentionally out-of-corpus — the support score should flip to `no`.

In [5]:
for qid in ['q01', 'q05', 'q11', 'q27']:
    q = QA[qid].question
    cands = retrieve(q, k=3)
    kept = [(d, t) for d, t in cands if grade(q, t)]
    body, supported = answer(q, kept)
    print(f'--- {qid} ---')
    print('question:', q)
    print('kept:', [d for d, _ in kept])
    print('answer:', body)
    print('supported:', supported)
    print()

--- q01 ---
question: By how much does RA-MoE reduce p99 decode latency compared to standard learned routing?
kept: ['synth-001']
answer: RA-MoE uses 47B total parameters with two experts active per token (~13B active params).
supported: True

--- q05 ---
question: What is the average length of documents in the long-form retrieval benchmark released with HA-Retrieve?
kept: []
answer: The Tokenizer-Equity Index (TEI) measures cross-lingual fairness by quantifying token-fertility imbalance.
supported: True

--- q11 ---
question: What is the Tokenizer Equity Index, and how many languages does the study evaluate?
kept: []
answer: Distill-Reason transfers chain-of-thought reasoning from large teachers to small students with ~80% of the gain at ~7% of the cost.
supported: True

--- q27 ---
question: What does the corpus report about ColBERT's late-interaction retrieval performance?
kept: []
answer: I don't know — none of the retrieved passages discuss this topic.
supported: False



## What you can extend

* Multi-pass: if `supported == no`, re-retrieve with a transformed query and re-answer.
* Per-chunk `support` scoring instead of one global flag — needed for citation UI.
* Replace the grader with a fine-tuned small classifier — way cheaper at scale.